In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Once mounted, your file paths in the config will change to something like:
# METADATA_PATH = "/content/drive/MyDrive/FIT3164/metadata_holistic_pose_hand.csv"

Mounted at /content/drive


In [6]:
# -*- coding: utf-8 -*-
"""Sign Language Recognition - Top 5 Glosses (Should Give High Accuracy)"""

import os
import json
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.optim as optim

# ==================== CONFIG ====================
ZIP_PATH = "/content/processed_holistic_pose_hand_top5.zip"   # Your new zip
METADATA_PATH = "/content/drive/MyDrive/metadata_holistic_pose_hand.csv"

EXTRACT_DIR = "/content"
NPY_DIR = "/content/processed_holistic_pose_hand_top5"        # Better folder name
LABEL_MAP_PATH = "/content/label_map_top5.json"
MODEL_SAVE_PATH = "/content/best_sign_lstm_top5.pth"

BATCH_SIZE = 32
NUM_EPOCHS = 40
LEARNING_RATE = 0.0008
WEIGHT_DECAY = 1e-4
MIN_VALID_FRAMES = 5
INPUT_SIZE = 225
HIDDEN_SIZE = 128          # Smaller model is enough for 5 classes
NUM_LAYERS = 2
DROPOUT = 0.3

NUM_CLASSES = 5            # <<< Now correctly set to 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}\n")

# ==================== EXTRACTION ====================
if not os.path.exists(NPY_DIR) or len(os.listdir(NPY_DIR)) < 10:
    if os.path.exists(ZIP_PATH) and zipfile.is_zipfile(ZIP_PATH):
        print("Extracting top5 zip file...")
        with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
            zip_ref.extractall(EXTRACT_DIR)
        print("✅ Extraction completed.")
    else:
        print(f"❌ Zip file not found: {ZIP_PATH}")
        raise SystemExit("Please check the zip path.")
else:
    print("✅ NPY data already exists.")

# ==================== DATASET ====================
class WLASLDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = np.load(row["colab_path"]).astype(np.float32)  # (30, 225)

        # Good normalization for small number of classes
        mean = x.mean(axis=0, keepdims=True)
        x = x - mean
        std = x.std(axis=0, keepdims=True) + 1e-8
        x = x / std

        if self.augment:
            x += np.random.normal(0, 0.012, x.shape).astype(np.float32)
            x *= np.random.uniform(0.95, 1.05)

            if np.random.rand() < 0.5:
                shift = np.random.randint(-3, 4)
                x = np.roll(x, shift, axis=0)

        y = int(row["label_id"])
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

# ==================== MODEL ====================
class SignLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True,
                            bidirectional=True, dropout=dropout if num_layers > 1 else 0)
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        self.attention = nn.Linear(hidden_size * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.layer_norm(out)
        attn = F.softmax(self.attention(out).squeeze(-1), dim=1)
        context = torch.sum(out * attn.unsqueeze(-1), dim=1)
        context = self.dropout(context)
        return self.fc(context)

# ==================== TRAIN / EVAL ====================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = total_correct = total_samples = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        total_correct += (torch.argmax(logits, dim=1) == y).sum().item()
        total_samples += x.size(0)
    return total_loss / total_samples, total_correct / total_samples


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = total_correct = total_samples = 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item() * x.size(0)
            total_correct += (torch.argmax(logits, dim=1) == y).sum().item()
            total_samples += x.size(0)
    return total_loss / total_samples, total_correct / total_samples

# ==================== MAIN ====================
def main():
    df = pd.read_csv(METADATA_PATH, dtype={"video_id": str})

    # === Filter to Top 5 Glosses ===
    gloss_counts = df['gloss'].value_counts()
    top_5_glosses = gloss_counts.nlargest(5).index.tolist()
    df = df[df['gloss'].isin(top_5_glosses)].reset_index(drop=True)

    print(f"Selected top 5 glosses: {top_5_glosses}")
    print(f"Total samples after filtering: {len(df)}")

    df = df[df["valid_frames"] >= MIN_VALID_FRAMES].reset_index(drop=True)

    df["colab_path"] = df["video_id"].astype(str).apply(
        lambda x: os.path.join(NPY_DIR, f"{x}.npy")
    )
    df = df[df["colab_path"].apply(os.path.exists)].reset_index(drop=True)

    # Create label mapping for only 5 classes
    glosses = sorted(df["gloss"].unique())
    label2id = {gloss: i for i, gloss in enumerate(glosses)}
    df["label_id"] = df["gloss"].map(label2id)

    with open(LABEL_MAP_PATH, "w", encoding="utf-8") as f:
        json.dump({"label2id": label2id, "id2label": {i: g for g, i in label2id.items()}},
                  f, ensure_ascii=False, indent=2)

    train_df = df[df["split"] == "train"].reset_index(drop=True)
    val_df   = df[df["split"] == "val"].reset_index(drop=True)
    test_df  = df[df["split"] == "test"].reset_index(drop=True)

    print(f"Final classes: {len(glosses)}")
    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}\n")

    train_loader = DataLoader(WLASLDataset(train_df, augment=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(WLASLDataset(val_df, augment=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(WLASLDataset(test_df, augment=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model = SignLSTM(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES, DROPOUT).to(DEVICE)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.7, patience=6, min_lr=1e-6)

    best_val_acc = 0.0
    patience = 12
    patience_counter = 0

    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)

        scheduler.step(val_acc)

        print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc + 0.005:
            best_val_acc = val_acc
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(f"   → Best model saved (Val Acc: {val_acc:.4f})")
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    print("\n" + "="*70)
    print("Loading best model for final test...")
    model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE))
    _, test_acc = evaluate(model, test_loader, criterion, DEVICE)
    print(f"FINAL TEST ACCURACY: {test_acc:.4f} ({test_acc*100:.2f}%)")
    print("="*70)

if __name__ == "__main__":
    main()

Using device: cuda

✅ NPY data already exists.
Selected top 5 glosses: ['computer', 'cousin', 'who', 'thin', 'basketball']
Total samples after filtering: 62
Final classes: 1
Train: 10 | Val: 3 | Test: 1

Epoch  1/40 | Train Acc: 0.2000 | Val Acc: 1.0000
   → Best model saved (Val Acc: 1.0000)
Epoch  2/40 | Train Acc: 1.0000 | Val Acc: 1.0000
Epoch  3/40 | Train Acc: 1.0000 | Val Acc: 1.0000
Epoch  4/40 | Train Acc: 1.0000 | Val Acc: 1.0000
Epoch  5/40 | Train Acc: 1.0000 | Val Acc: 1.0000
Epoch  6/40 | Train Acc: 1.0000 | Val Acc: 1.0000
Epoch  7/40 | Train Acc: 1.0000 | Val Acc: 1.0000
Epoch  8/40 | Train Acc: 1.0000 | Val Acc: 1.0000
Epoch  9/40 | Train Acc: 1.0000 | Val Acc: 1.0000
Epoch 10/40 | Train Acc: 1.0000 | Val Acc: 1.0000
Epoch 11/40 | Train Acc: 1.0000 | Val Acc: 1.0000
Epoch 12/40 | Train Acc: 1.0000 | Val Acc: 1.0000
Epoch 13/40 | Train Acc: 1.0000 | Val Acc: 1.0000
Early stopping triggered.

Loading best model for final test...
FINAL TEST ACCURACY: 1.0000 (100.00%)


In [8]:
# -*- coding: utf-8 -*-
"""Fixed Top 5 Glosses Version - Reliable Training"""

import os
import json
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.optim as optim

# ==================== CONFIG ====================
ZIP_PATH = "/content/processed_holistic_pose_hand_top5.zip"
METADATA_PATH = "/content/drive/MyDrive/metadata_holistic_pose_hand.csv"

EXTRACT_DIR = "/content"
NPY_DIR = "/content/processed_holistic_pose_hand_top5"
LABEL_MAP_PATH = "/content/label_map_top5.json"
MODEL_SAVE_PATH = "/content/best_sign_lstm_top5_fixed.pth"

BATCH_SIZE = 16          # Smaller batch for tiny dataset
NUM_EPOCHS = 30
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-5
INPUT_SIZE = 225
HIDDEN_SIZE = 96         # Small model for 5 classes
NUM_LAYERS = 2
DROPOUT = 0.2

NUM_CLASSES = 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}\n")

# ==================== EXTRACTION ====================
if not os.path.exists(NPY_DIR):
    if os.path.exists(ZIP_PATH) and zipfile.is_zipfile(ZIP_PATH):
        print("Extracting top5 zip...")
        with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
            zip_ref.extractall(EXTRACT_DIR)
        print("✅ Extraction completed.")
    else:
        print("❌ Zip not found!")
        raise SystemExit("Check ZIP_PATH")
else:
    print("✅ NPY data ready.")

# ==================== DATASET ====================
class WLASLDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = np.load(row["colab_path"]).astype(np.float32)

        mean = x.mean(axis=0, keepdims=True)
        x = x - mean
        std = x.std(axis=0, keepdims=True) + 1e-8
        x = x / std

        if self.augment and len(self.df) > 10:   # only augment if enough data
            x += np.random.normal(0, 0.01, x.shape).astype(np.float32)
            x *= np.random.uniform(0.95, 1.05)

        y = int(row["label_id"])
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

# ==================== MODEL ====================
class SignLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True,
                            bidirectional=True, dropout=dropout if num_layers > 1 else 0)
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        self.attention = nn.Linear(hidden_size * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.layer_norm(out)
        attn = F.softmax(self.attention(out).squeeze(-1), dim=1)
        context = torch.sum(out * attn.unsqueeze(-1), dim=1)
        context = self.dropout(context)
        return self.fc(context)

# ==================== TRAIN / EVAL (same as before) ====================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = total_correct = total_samples = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        total_correct += (torch.argmax(logits, dim=1) == y).sum().item()
        total_samples += x.size(0)
    return total_loss / total_samples, total_correct / total_samples

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = total_correct = total_samples = 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item() * x.size(0)
            total_correct += (torch.argmax(logits, dim=1) == y).sum().item()
            total_samples += x.size(0)
    return total_loss / total_samples, total_correct / total_samples

# ==================== MAIN - FIXED ====================
def main():
    df = pd.read_csv(METADATA_PATH, dtype={"video_id": str})

    # Force top 5 glosses
    target_glosses = ['computer', 'cousin', 'who', 'thin', 'basketball']
    df = df[df['gloss'].isin(target_glosses)].reset_index(drop=True)

    print(f"Selected glosses: {target_glosses}")
    print(f"Total samples: {len(df)}")
    print(f"Distribution:\n{df['gloss'].value_counts()}\n")

    df = df[df["valid_frames"] >= MIN_VALID_FRAMES].reset_index(drop=True)

    df["colab_path"] = df["video_id"].astype(str).apply(
        lambda x: os.path.join(NPY_DIR, f"{x}.npy")
    )
    df = df[df["colab_path"].apply(os.path.exists)].reset_index(drop=True)

    # Create label mapping
    glosses = sorted(df["gloss"].unique())
    label2id = {gloss: i for i, gloss in enumerate(glosses)}
    df["label_id"] = df["gloss"].map(label2id)

    print(f"Final number of classes: {len(glosses)} → {glosses}\n")

    with open(LABEL_MAP_PATH, "w", encoding="utf-8") as f:
        json.dump({"label2id": label2id, "id2label": {i: g for g, i in label2id.items()}},
                  f, ensure_ascii=False, indent=2)

    train_df = df[df["split"] == "train"].reset_index(drop=True)
    val_df = df[df["split"] == "val"].reset_index(drop=True)
    test_df = df[df["split"] == "test"].reset_index(drop=True)

    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}\n")

    if len(train_df) < 5 or len(test_df) < 1:
        print("WARNING: Too few samples in splits. Results will not be reliable.")

    train_loader = DataLoader(WLASLDataset(train_df, augment=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    val_loader   = DataLoader(WLASLDataset(val_df, augment=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    test_loader  = DataLoader(WLASLDataset(test_df, augment=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    model = SignLSTM(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES, DROPOUT).to(DEVICE)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.7, patience=5, min_lr=1e-6)

    best_val_acc = 0.0
    patience = 10
    patience_counter = 0

    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)

        scheduler.step(val_acc)

        print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc + 0.01:
            best_val_acc = val_acc
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(f"   → Best model saved")
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    print("\n" + "="*70)
    model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE))
    _, test_acc = evaluate(model, test_loader, criterion, DEVICE)
    print(f"FINAL TEST ACCURACY: {test_acc:.4f} ({test_acc*100:.2f}%)")
    print("="*70)

if __name__ == "__main__":
    main()

Using device: cuda

✅ NPY data ready.
Selected glosses: ['computer', 'cousin', 'who', 'thin', 'basketball']
Total samples: 62
Distribution:
gloss
computer      14
cousin        13
who           12
thin          12
basketball    11
Name: count, dtype: int64

Final number of classes: 1 → ['computer']

Train: 10 | Val: 3 | Test: 1

Epoch  1/30 | Train Acc: 0.6000 | Val Acc: 1.0000
   → Best model saved
Epoch  2/30 | Train Acc: 1.0000 | Val Acc: 1.0000
Epoch  3/30 | Train Acc: 1.0000 | Val Acc: 1.0000
Epoch  4/30 | Train Acc: 1.0000 | Val Acc: 1.0000
Epoch  5/30 | Train Acc: 1.0000 | Val Acc: 1.0000
Epoch  6/30 | Train Acc: 1.0000 | Val Acc: 1.0000
Epoch  7/30 | Train Acc: 1.0000 | Val Acc: 1.0000
Epoch  8/30 | Train Acc: 1.0000 | Val Acc: 1.0000
Epoch  9/30 | Train Acc: 1.0000 | Val Acc: 1.0000
Epoch 10/30 | Train Acc: 1.0000 | Val Acc: 1.0000
Epoch 11/30 | Train Acc: 1.0000 | Val Acc: 1.0000
Early stopping triggered.

FINAL TEST ACCURACY: 1.0000 (100.00%)
